In [34]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "dbrepo==1.13.3"],
               capture_output=True)

from dbrepo.RestClient import RestClient
from dbrepo.api.dto import QueryDefinition, FilterDefinition, FilterType
import os
import requests

print("Libraries ready!")

Libraries ready!


In [35]:
os.environ["DBREPO_PASSWORD"] = input("Enter your DBRepo password: ")

client = RestClient(
    endpoint="https://test.dbrepo.tuwien.ac.at",
    username="12534814",
    password=os.environ.get("DBREPO_PASSWORD")
)

DATABASE_ID = "13457a52-37f9-48d4-a078-6865e8d35981"
print("Connected:", client.whoami())

12534814
Connected: 12534814


In [36]:
tables = client.get_tables(database_id=DATABASE_ID)
table_lookup = {}
for table in tables:
    full = client.get_table(database_id=DATABASE_ID, table_id=table.id)
    table_lookup[table.name] = {
        "table_id": table.id,
        "columns": {col.name: col.id for col in full.columns}
    }
    print(f"{table.name} | internal: {table.internal_name}")

wqm_table = client.get_table(
    database_id=DATABASE_ID,
    table_id=table_lookup["water_quality_measurement"]["table_id"]
)
print(f"\nwater_quality_measurement has {len(wqm_table.columns)} columns")

water_quality_measurement | internal: water_quality_measurement
sampling_event | internal: sampling_event
sampling_station | internal: sampling_station
lake | internal: lake

water_quality_measurement has 108 columns


In [37]:
views_to_create = [
    {
        "name": "core_water_quality_features",
        "description": "Core physicochemical features for ML pipeline",
        "col_names": [
            "measurement_id", "vandens_temp", "suspend_medziagos",
            "sarmingumas", "deguonis_istirpes", "ph", "skaidrumas",
            "elektr_laidis", "biochem_deg_suvartojimas", "amonio_azotas",
            "nitritu_azotas", "nitratu_azotas", "azotas_mineralinis",
            "azotas_bendras", "fosfatu_fosforas", "fosforas_bendras",
            "anglingumas", "chlorofilas_a", "kalcio_karbonatas"
        ]
    },
    {
        "name": "eutrophication_risk_indicators",
        "description": "Nutrients and chlorophyll for eutrophication risk classification",
        "col_names": [
            "measurement_id", "chlorofilas_a", "fosforas_bendras",
            "azotas_bendras", "ph", "skaidrumas", "deguonis_istirpes"
        ]
    },
    {
        "name": "heavy_metal_pollution_features",
        "description": "Heavy metal concentrations for pollution analysis",
        "col_names": [
            "measurement_id", "gyvsidabris", "kadmis", "nikelis",
            "svinas", "varis", "chromas", "vanadis", "aliuminis",
            "alavas", "arsenas", "cinkas"
        ]
    },
    {
        "name": "nutrient_pollution_features",
        "description": "Nitrogen and phosphorus indicators for water quality classification",
        "col_names": [
            "measurement_id", "amonio_azotas", "nitritu_azotas",
            "nitratu_azotas", "azotas_mineralinis", "azotas_bendras",
            "fosfatu_fosforas", "fosforas_bendras", "chlorofilas_a"
        ]
    }
]

for v in views_to_create:
    try:
        cols = [
            f"water_quality_measurement.{col.name}"
            for col in wqm_table.columns
            if col.name in v["col_names"]
        ]
        view = client.create_view(
            database_id=DATABASE_ID,
            name=v["name"],
            query=QueryDefinition(
                columns=cols,
                datasources=["water_quality_measurement"]
            ),
            is_public=True,
            is_schema_public=True
        )
        print(f"✓ Created: {v['name']} | ID: {view.id}")
    except Exception as e:
        if "name exists" in str(e):
            print(f"⚠ Already exists: {v['name']} — skipping")
        else:
            print(f"✗ Error: {v['name']} → {e}")

⚠ Already exists: core_water_quality_features — skipping
⚠ Already exists: eutrophication_risk_indicators — skipping
⚠ Already exists: heavy_metal_pollution_features — skipping
⚠ Already exists: nutrient_pollution_features — skipping


In [38]:
views = client.get_views(database_id=DATABASE_ID)
print(f"Views in DBRepo: {len(views)}")
for v in views:
    print(f"  ✓ {v.name} | ID: {v.id}")

Views in DBRepo: 4
  ✓ nutrient_pollution_features | ID: c8cae9de-654c-4bc2-be19-0e2d59798c04
  ✓ heavy_metal_pollution_features | ID: 371016af-2971-406a-900f-3c1234d584e8
  ✓ eutrophication_risk_indicators | ID: 69cb3873-7e7c-4417-925c-3615a4e3c220
  ✓ core_water_quality_features | ID: a91dc02b-457a-4dba-8f46-8386eee8484a
